# GAT Attention-Based Structure-Activity Analysis

**Goal.** Use the trained GAT model's learned attention weights to identify which
neighboring molecules most influence a test-set compound being classified as **AI**
(IDO-selective) or **IA** (TDO-selective), then extract the structural motifs
(Murcko scaffolds, maximum common substructures) shared by those influential
neighbors. This turns the attention mechanism — which GAT already computes as part
of prediction — into a source of interpretable structure-activity insight, directly
addressing the reviewer's concern that the study lacked "in-depth analyses of
candidate compounds, structure-activity relationships, and pharmacological
implications."

**How this works, conceptually.** In your similarity graph, an edge between molecule
A and molecule B exists because they are structurally similar (Tanimoto similarity on
Morgan fingerprints). GAT's attention mechanism *re-weights* these edges per
prediction — it learns, for each node, which of its similar neighbors are actually
most informative for classifying it correctly. So: for a molecule GAT correctly
classifies as AI, the neighbors it pays the **most attention to** are, by
construction, other molecules whose structure the model has learned is most
diagnostic of IDO-selectivity. Aggregating those high-attention neighbors across many
correctly-classified AI molecules, then asking "what structural motif do they share,"
gives you a data-driven, model-grounded hypothesis about which substructures drive
IDO- vs. TDO-selectivity — a genuine SAR contribution, not just a benchmarking result.

**Adapted to your actual `RF_GAT.ipynb`.** This version matches your real
`GAT_EdgeWeighted` architecture: **three** `GATConv` layers (4 heads -> 2 heads ->
1 head), each with a linear skip connection, and `BatchNorm` + ELU after layers 1
and 2 (layer 3 has no activation, feeding directly into the classifier). Section 2
below uses the same `model` / `test_data` / `df1['smiles']` variables you already
have defined in `RF_GAT.ipynb` (cell 86) — copy these cells in after that point in
your notebook, or run this notebook after re-running `RF_GAT.ipynb` up through that
cell so those variables are in scope.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import networkx as nx

from rdkit import Chem
from rdkit.Chem import Draw, rdFMCS
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.Draw import rdMolDraw2D

from collections import Counter

%matplotlib inline
CLASS_NAMES = ["AA", "AI", "IA", "II"]  # 0=AA, 1=AI, 2=IA, 3=II — matches the rest of your project

## 2. Model, test data, and SMILES mapping

These are the same variables already defined in your `RF_GAT.ipynb` (cell 86):
`model` (trained `GAT_EdgeWeighted`, checkpoint `best_gat_inductive.pt` already
loaded, in `.eval()` mode), `test_data` (the inductive test `Data` object,
containing `.x`, `.edge_index`, `.edge_weight`, `.y`, `.test_mask`), and
`node_to_smiles` (from `df1['smiles']`, indexed the same way as your graph nodes).
If you're running this as a continuation of `RF_GAT.ipynb` in the same kernel, this
cell just confirms they're in scope; otherwise, re-run the equivalent lines here.

In [ ]:
# Confirms the variables from RF_GAT.ipynb (cell 86) are in scope.
# If running as a fresh kernel rather than a continuation, uncomment and adapt:
#
# model = GAT_EdgeWeighted(num_features=train_data.x.size(1),
#                           num_classes=int(train_data.y.max()) + 1).to(device)
# model.load_state_dict(torch.load("best_gat_inductive.pt", map_location=device))
# model.eval()
# data = test_data
# node_to_smiles = df1["smiles"].to_dict()

assert model is not None and data is not None and node_to_smiles is not None
model.eval()
print(f"Model ready. Test graph: {data.num_nodes} nodes, "
      f"{data.test_mask.sum().item()} test-set molecules, "
      f"{len(node_to_smiles)} SMILES mapped.")

## 3. Extract attention weights from the trained model

This manually re-runs your GAT's actual 3-layer forward pass — `conv1` (4 heads) ->
skip1 -> BatchNorm -> ELU -> `conv2` (2 heads) -> skip2 -> BatchNorm -> ELU ->
`conv3` (1 head) -> skip3 — requesting `return_attention_weights=True` from each
`GATConv` so we capture attention at every layer without changing the model's actual
computation (the returned hidden states are identical to what `model.forward()`
would produce; we're just also capturing the attention coefficients along the way).

`layer` selects which of the 3 layers' attention to use: `"last"` (layer 3, the one
feeding directly into the classifier — most directly tied to the final prediction),
`"first"` (layer 1, closest to raw molecular similarity), or `"middle"` (layer 2).

As before, `GATConv`'s self-loop edges (added automatically since
`add_self_loops=True` in your architecture) are filtered out of the result so they
aren't mistaken for real neighbor molecules.

In [ ]:
@torch.no_grad()
def get_attention_weights(model, data, layer="last"):
    """
    Returns (edge_index, alpha) from one GATConv layer of `model`, with self-loop
    edges removed. alpha is averaged across attention heads.

    Mirrors GAT_EdgeWeighted.forward() exactly (3 GATConv layers, each with a
    linear skip connection; BatchNorm + ELU after layers 1 and 2; no activation
    after layer 3), so the hidden states computed here match what forward() itself
    would produce -- we are only additionally capturing each layer's attention
    coefficients along the way.
    """
    model.eval()
    x, edge_index, edge_weight = data.x, data.edge_index, data.edge_weight
    edge_attr = edge_weight.view(-1, 1) if edge_weight is not None else None

    # ---- Layer 1 ----
    residual = model.skip1(x) if model.skip1 is not None else x
    h1, (ei1, alpha1) = model.conv1(x, edge_index, edge_attr=edge_attr,
                                      return_attention_weights=True)
    h1 = h1 + residual
    h1 = model.bn1(h1)
    h1 = torch.nn.functional.elu(h1)
    h1 = model.dropout(h1)  # no-op in eval mode

    # ---- Layer 2 ----
    residual = model.skip2(h1)
    h2, (ei2, alpha2) = model.conv2(h1, edge_index, edge_attr=edge_attr,
                                      return_attention_weights=True)
    h2 = h2 + residual
    h2 = model.bn2(h2)
    h2 = torch.nn.functional.elu(h2)
    h2 = model.dropout(h2)  # no-op in eval mode

    # ---- Layer 3 (no activation) ----
    residual = model.skip3(h2)
    h3, (ei3, alpha3) = model.conv3(h2, edge_index, edge_attr=edge_attr,
                                      return_attention_weights=True)
    h3 = h3 + residual

    layer_map = {"first": (ei1, alpha1), "middle": (ei2, alpha2), "last": (ei3, alpha3)}
    ei, alpha = layer_map[layer]
    alpha = alpha.mean(dim=1)  # average across heads -> [E']

    # remove self-loops (src == dst)
    src, dst = ei[0], ei[1]
    keep = src != dst
    return ei[:, keep], alpha[keep]


edge_index_att, alpha = get_attention_weights(model, data, layer="last")
print(f"Extracted attention for {edge_index_att.shape[1]} non-self-loop edges "
      f"(range: {alpha.min():.4f} - {alpha.max():.4f})")

## 4. Top-attention neighbors per correctly-classified AI / IA test molecule

For every test-set molecule whose **true label matches its predicted label** for AI
or IA (so we're explaining genuinely correct predictions, not the model's mistakes),
find its top-*k* neighbors by attention weight in the graph.

In [ ]:
TOP_K = 3
TARGET_CLASSES = {"AI": 1, "IA": 2}  # class name -> label index, per this project's convention

@torch.no_grad()
def get_predictions(model, data):
    """ADAPT if your forward() returns something other than (out, h)."""
    out, h = model(data.x, data.edge_index, data.edge_weight)
    return out.argmax(dim=1)

y_pred = get_predictions(model, data)
y_true = data.y
test_mask = data.test_mask

src, dst = edge_index_att[0], edge_index_att[1]

def top_k_neighbors(node_idx, k=TOP_K):
    """Returns [(neighbor_node_idx, attention_weight), ...] sorted descending,
    considering edges where node_idx is the destination (i.e. neighbors attending
    INTO this node — this is what the model actually uses to form node_idx's
    representation)."""
    mask = dst == node_idx
    neigh_nodes = src[mask]
    neigh_alpha = alpha[mask]
    if len(neigh_alpha) == 0:
        return []
    order = torch.argsort(neigh_alpha, descending=True)[:k]
    return [(neigh_nodes[i].item(), neigh_alpha[i].item()) for i in order]

records = []
for cls_name, cls_idx in TARGET_CLASSES.items():
    test_indices = torch.where(test_mask & (y_true == cls_idx) & (y_pred == cls_idx))[0]
    for node_idx in test_indices.tolist():
        for neigh_idx, w in top_k_neighbors(node_idx):
            records.append({
                "class": cls_name,
                "target_node": node_idx,
                "target_smiles": node_to_smiles.get(node_idx, None),
                "neighbor_node": neigh_idx,
                "neighbor_smiles": node_to_smiles.get(neigh_idx, None),
                "attention_weight": w,
            })

attn_df = pd.DataFrame(records)
print(f"{len(attn_df)} (target, neighbor) attention edges collected "
      f"across {attn_df['target_node'].nunique()} correctly-classified AI/IA molecules")
attn_df.head(10)

## 5. Structural motif analysis of the influential neighbors

For each class (AI, IA), pool all the top-attention neighbor molecules and ask:
**what structural motif do they share?** Two complementary views:

1. **Murcko scaffold frequency** — the core ring-system scaffold of each neighbor,
   tallied. A scaffold that recurs across many independent target molecules' top
   neighbors is a strong candidate for a class-discriminating motif.
2. **Maximum Common Substructure (MCS)** across the most frequent neighbors — a
   single consensus substructure, useful as a figure.

In [ ]:
def scaffold_frequency(smiles_list, top_n=10):
    scaffolds = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        try:
            scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
        except Exception:
            continue
        if scaf:  # skip empty scaffolds (fully acyclic molecules)
            scaffolds.append(scaf)
    counts = Counter(scaffolds)
    return counts.most_common(top_n)


scaffold_results = {}
for cls_name in TARGET_CLASSES:
    neighbor_smiles = attn_df.loc[attn_df["class"] == cls_name, "neighbor_smiles"].dropna().tolist()
    scaffold_results[cls_name] = scaffold_frequency(neighbor_smiles)
    print(f"\n=== Top scaffolds among {cls_name} top-attention neighbors "
          f"(n={len(neighbor_smiles)} neighbor mentions) ===")
    for scaf, count in scaffold_results[cls_name]:
        print(f"  {count:3d}x  {scaf}")

In [ ]:
def top_neighbors_mcs(smiles_list, n_mols=8):
    """MCS across the n_mols most attention-weighted, valid, distinct neighbor molecules."""
    mols = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            mols.append(mol)
        if len(mols) >= n_mols:
            break
    if len(mols) < 2:
        return None
    return rdFMCS.FindMCS(mols, ringMatchesRingOnly=True, completeRingsOnly=True,
                           timeout=30)

for cls_name in TARGET_CLASSES:
    sub = attn_df[attn_df["class"] == cls_name].sort_values("attention_weight", ascending=False)
    smiles_ranked = sub["neighbor_smiles"].dropna().drop_duplicates().tolist()
    mcs = top_neighbors_mcs(smiles_ranked)
    if mcs is not None:
        print(f"{cls_name} MCS ({mcs.numAtoms} atoms, {mcs.numBonds} bonds): {mcs.smartsString}")
    else:
        print(f"{cls_name}: not enough valid distinct neighbor molecules for MCS")

## 6. Visualization

### 6a. Top influential neighbor molecules, per class (manuscript figure candidate)

In [ ]:
def draw_top_neighbors(cls_name, n=6):
    sub = (attn_df[attn_df["class"] == cls_name]
           .sort_values("attention_weight", ascending=False)
           .drop_duplicates(subset="neighbor_smiles")
           .head(n))
    mols, legends = [], []
    for _, row in sub.iterrows():
        mol = Chem.MolFromSmiles(row["neighbor_smiles"])
        if mol is not None:
            mols.append(mol)
            legends.append(f"attn={row['attention_weight']:.3f}")
    if not mols:
        print(f"No valid neighbor molecules found for class {cls_name} "
              f"(check that enough test molecules of this class were correctly "
              f"classified, and that TOP_K / test set size are large enough).")
        return None
    return Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(250, 250), legends=legends)


def save_grid_image(img, path):
    """
    Draw.MolsToGridImage returns a PIL Image in most RDKit setups, but an
    IPython.display.Image (which has .data bytes, not .save()) when RDKit
    detects a notebook/IPythonConsole environment. Handle both.
    """
    if img is None:
        return
    if hasattr(img, "save"):          # PIL.Image
        img.save(path)
    elif hasattr(img, "data"):         # IPython.display.Image
        with open(path, "wb") as f:
            f.write(img.data)
    else:
        raise TypeError(f"Unrecognized image type from MolsToGridImage: {type(img)}")

img_ai = draw_top_neighbors("AI")
save_grid_image(img_ai, "top_attention_neighbors_AI.png")
img_ai

In [ ]:
img_ia = draw_top_neighbors("IA")
save_grid_image(img_ia, "top_attention_neighbors_IA.png")
img_ia

### 6b. Most common scaffold, highlighted (manuscript figure candidate)

In [ ]:
def draw_top_scaffold(cls_name):
    top_scaffold_smiles = scaffold_results[cls_name][0][0]
    scaffold_mol = Chem.MolFromSmiles(top_scaffold_smiles)
    drawer = rdMolDraw2D.MolDraw2DCairo(400, 400)
    drawer.DrawMolecule(scaffold_mol)
    drawer.FinishDrawing()
    with open(f"top_scaffold_{cls_name}.png", "wb") as f:
        f.write(drawer.GetDrawingText())
    return scaffold_mol

for cls_name in TARGET_CLASSES:
    if scaffold_results[cls_name]:
        mol = draw_top_scaffold(cls_name)
        print(f"{cls_name} most common scaffold saved: top_scaffold_{cls_name}.png "
              f"(seen {scaffold_results[cls_name][0][1]}x)")

### 6c. Attention subgraph for one example molecule (illustrates the method itself — good for a methods figure)

In [ ]:
def plot_attention_subgraph(node_idx, k=TOP_K, ax=None):
    neighbors = top_k_neighbors(node_idx, k=k)
    if not neighbors:
        print("No incoming attention edges for this node.")
        return

    G = nx.DiGraph()
    G.add_node(node_idx, label="target")
    for neigh_idx, w in neighbors:
        G.add_node(neigh_idx, label="neighbor")
        G.add_edge(neigh_idx, node_idx, weight=w)

    pos = nx.spring_layout(G, seed=0)
    ax = ax or plt.gca()
    weights = [G[u][v]["weight"] * 12 for u, v in G.edges()]
    nx.draw_networkx_nodes(G, pos, nodelist=[node_idx], node_color="#2563eb",
                             node_size=900, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=[n for n, _ in neighbors], node_color="#94a3b8",
                             node_size=600, ax=ax)
    nx.draw_networkx_edges(G, pos, width=weights, edge_color="#475569",
                             arrows=True, arrowsize=15, ax=ax)
    labels = {node_idx: f"target\n(node {node_idx})"}
    labels.update({n: f"node {n}\nattn={w:.3f}" for n, w in neighbors})
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, ax=ax)
    ax.set_title(f"Top-{k} attention-weighted neighbors of node {node_idx}")
    ax.axis("off")

# example: first correctly-classified AI test molecule
example_node = attn_df.loc[attn_df["class"] == "AI", "target_node"].iloc[0]
fig, ax = plt.subplots(figsize=(6, 5))
plot_attention_subgraph(example_node, ax=ax)
plt.tight_layout()
plt.savefig("example_attention_subgraph.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Save summary tables for the manuscript

In [ ]:
attn_df.to_csv("gat_attention_neighbors_AI_IA.csv", index=False)

scaffold_summary_rows = []
for cls_name, results in scaffold_results.items():
    for rank, (scaf, count) in enumerate(results, start=1):
        scaffold_summary_rows.append({"class": cls_name, "rank": rank,
                                        "scaffold_smiles": scaf, "count": count})
pd.DataFrame(scaffold_summary_rows).to_csv("gat_attention_scaffold_summary.csv", index=False)

print("Saved: gat_attention_neighbors_AI_IA.csv, gat_attention_scaffold_summary.csv")

## 8. Suggested next step: cross-reference against known pharmacophores

The scaffold and MCS results above are data-driven hypotheses from your model, not
yet validated pharmacology. A strong revision would add one more step: compare the
top recurring AI/IA scaffolds against **published IDO1/TDO pharmacophore or binding
mode literature** — e.g., known heme-iron-coordinating groups, selectivity-determining
substituents reported in prior IDO1/TDO SAR studies — and discuss whether your
model's attention-derived motifs recover known chemistry (which would validate the
approach) or surface a potentially novel selectivity-determining motif (which would
be a genuine finding worth highlighting to reviewers). This is chemistry-domain
judgment that needs your (or a medicinal chemist co-author's) expertise — I can help
structure that comparison/discussion once you have candidate literature SMARTS
patterns or reference compounds to compare against.